# Reproducing Figure 3 of LearnPruner (ICLR 2026)

Interactive front-end for `vtp_eval.figure3.*`. Same logic as the CLI (`python -m vtp_eval.figure3`).

## Insight (Figure 3)
> *"In both shallow and deep layers, uninformative regions tend to absorb most of the attention from text tokens, while in the middle layer, different text tokens are able to focus on their semantic-related regions, hence allowing for effective token selection."*

## Workflow
1. Run **Cell 1** to download COCO annotations and surface 8 candidate images with their full category lists.
2. Pick a `CHOSEN_INDEX` and the `WORDS` to track from that row's categories. Re-run **Cell 2**.
3. Run the remaining cells: load LLaVA, extract attention, render Figure 3, compute concentration metrics.

In [ ]:
# Cell 0 — environment
import sys, torch
print('Python:', sys.version.split()[0], '| torch:', torch.__version__,
      '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA GPU required.'

## 1. Surface candidates with full COCO annotations

In [ ]:
# Cell 1 — candidates
from pathlib import Path
from vtp_eval.figure3.coco import (
    DEFAULT_ANN_DIR, DEFAULT_ROOT,
    ensure_coco_ann, pick_candidates,
    save_candidates_preview, dump_candidates_table,
)

OUT_DIR = (DEFAULT_ROOT / 'outputs') if DEFAULT_ROOT.name == 'workspace' else Path('./outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

ann_json = ensure_coco_ann(DEFAULT_ANN_DIR)
CANDS, IMG_INFO = pick_candidates(ann_json, min_cats=3, max_cats=6, n=8)
save_candidates_preview(CANDS, IMG_INFO, OUT_DIR / 'coco_candidates.png')
dump_candidates_table(CANDS, OUT_DIR / 'candidates.json')

from PIL import Image
Image.open(OUT_DIR / 'coco_candidates.png')

## 2. Pick image + target words

Read the table printed by Cell 1, choose an `idx`, and set `WORDS` to category names from that row. Multi-word categories (`'sports ball'`, `'tennis racket'`) are supported.

In [ ]:
# Cell 2 — pick
from vtp_eval.figure3.coco import download_candidate_image
from vtp_eval.figure3.tokens import build_default_query
from PIL import Image

CHOSEN_INDEX = 0
WORDS        = ['person', 'sports ball']     # ← pick from the categories printed above
QUERY        = None                           # None ⇒ auto-build from WORDS

chosen = CANDS[CHOSEN_INDEX]
IMAGE  = Image.open(download_candidate_image(chosen['iid'], IMG_INFO)).convert('RGB')
IMAGE.save(OUT_DIR / 'chosen_image.jpg')
QUERY = QUERY or build_default_query(WORDS)
print(f'idx={CHOSEN_INDEX}  id={chosen["iid"]}  size={IMAGE.size}')
print('Categories:', [n for n, _ in chosen['categories']])
print('Target words:', WORDS)
print('Query:', repr(QUERY))
IMAGE

## 3. Load LLaVA-1.5-7B + extract attention

In [ ]:
# Cell 3 — model + forward + attention extraction
from vtp_eval.figure3.attention import load_llava, extract_attention
MODEL, PROC = load_llava('llava-hf/llava-1.5-7b-hf')
PWL, WORD_POS, GRID, SINKS, LYRS = extract_attention(
    MODEL, PROC, IMAGE, QUERY, WORDS, layers=[2, 12, 30])
print('Word positions:', WORD_POS)
print('Sinks:', sorted(SINKS))
print('LYRS:', LYRS)

## 4. Figure 3 — heatmap overlay grid

In [ ]:
# Cell 4 — heatmap grid
from vtp_eval.figure3.visualize import plot_heatmap_grid
plot_heatmap_grid(PWL, IMAGE, SINKS, GRID, LYRS, QUERY,
                  OUT_DIR / 'figure3_reproduction.png')
from PIL import Image
Image.open(OUT_DIR / 'figure3_reproduction.png')

## 5. Concentration metrics + bar chart

In [ ]:
# Cell 5 — metrics
from vtp_eval.figure3.metrics import compute_metrics
from vtp_eval.figure3.visualize import plot_metrics_bar
DF = compute_metrics(PWL, SINKS, LYRS)
DF.to_csv(OUT_DIR / 'figure3_metrics.csv', index=False)
print(DF.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
plot_metrics_bar(DF, LYRS, OUT_DIR / 'figure3_metrics.png')
from PIL import Image
Image.open(OUT_DIR / 'figure3_metrics.png')